# La carte d'identité de chaque fichier
Avec panda, il va s'agir de d'examiner les fichiers .csv du dossier /datas, et pour chacun, générer leur carte d'identité dans un fichier artifacts/ingestion/identities/<AAAAMMJJ>/<nom du fichier analysé>_<HHMMSS>.md

Chaque fichier sera constitué de la manière suivante :
* Source: <chemin/complet/vers/le/document/source>
* Filsize : 9999.99 [Ko | Mo | ....]
* ProcessBeginAt: HH:MM:SS
* ProcessEndAt: HH:MM:SS
* Charset: [UTF8 | ISO8859 | Latin | ...]
* Rows Number : 9999
* Cols Number : 9999
* Column list (idéalement dans un tableau) | Colname | Infered Type | | --------| -------------| | machine_id | IDENTITY |

Pour chaque colonne identifiée numérique ou date :
* Col name,
* Min value,
* Max value,
* Mean value (uniquement pour les numériques),
* Median value (uniquement pour les numériques)

Affiche un compte-rendu en fin de traitement, et arrête-toi en cas d'erreur, en affichant le descriptif de l'erreur.

In [ ]:
from __future__ import annotations

import csv
import re
from dataclasses import dataclass
from datetime import datetime
from pathlib import Path
from typing import Any

import pandas as pd
from pandas.api.types import is_bool_dtype, is_numeric_dtype


@dataclass(frozen=True)
class IdentityResult:
    source: Path
    artifact: Path
    rows: int
    columns: int


def detect_charset(path: Path) -> tuple[str, str]:
    """Retourne le nom lisible et le codec Python du fichier."""
    content = path.read_bytes()
    candidates = (
        ("UTF-8 BOM", "utf-8-sig"),
        ("UTF-8", "utf-8"),
        ("Windows-1252", "cp1252"),
        ("ISO-8859-1", "iso-8859-1"),
    )

    if content.startswith(b"\xef\xbb\xbf"):
        return candidates[0]

    for label, codec in candidates[1:]:
        try:
            content.decode(codec)
            return label, codec
        except UnicodeDecodeError:
            continue

    raise UnicodeError(f"Impossible de déterminer le charset de {path}")


def detect_delimiter(path: Path, encoding: str) -> str:
    with path.open("r", encoding=encoding, newline="") as source:
        sample = source.read(16_384)
    if not sample:
        raise ValueError("Le fichier est vide.")
    try:
        return csv.Sniffer().sniff(sample, delimiters=",;\t|").delimiter
    except csv.Error as exc:
        raise ValueError("Séparateur CSV indéterminable.") from exc


def human_size(byte_count: int) -> str:
    units = ("o", "Ko", "Mo", "Go", "To")
    size = float(byte_count)
    for unit in units:
        if size < 1024 or unit == units[-1]:
            return f"{size:.2f} {unit}"
        size /= 1024
    raise AssertionError("Unité de taille inaccessible")


def _non_null_text(series: pd.Series) -> pd.Series:
    return series.dropna().astype("string").str.strip()


def infer_column_type(name: str, series: pd.Series) -> tuple[str, pd.Series | None]:
    """Infère un type métier et renvoie éventuellement les valeurs temporelles."""
    normalized_name = name.casefold().strip()
    values = _non_null_text(series)

    if is_bool_dtype(series.dtype):
        return "BOOLEAN", None

    date_hint = bool(
        re.search(r"(^|_)(date|datetime|timestamp|created_at|updated_at)($|_)", normalized_name)
    )
    date_shape = bool(
        not values.empty
        and values.str.match(r"^\d{4}-\d{2}-\d{2}(?:[ T].*)?$").mean() >= 0.90
    )
    if date_hint or date_shape:
        parsed = pd.to_datetime(values, errors="coerce", format="mixed")
        if not values.empty and parsed.notna().mean() >= 0.90:
            inferred = "DATETIME" if values.str.contains(r"[ T]\d{2}:\d{2}", regex=True).any() else "DATE"
            return inferred, parsed

    time_hint = normalized_name == "time" or normalized_name.endswith("_time")
    time_shape = bool(
        not values.empty
        and values.str.match(r"^\d{2}:\d{2}(?::\d{2}(?:\.\d+)?)?$").mean() >= 0.90
    )
    if time_hint and time_shape:
        return "TIME", values

    identity_hint = bool(
        normalized_name == "id"
        or normalized_name.endswith("_id")
        or any(token in normalized_name for token in ("badge", "identifier", "identifiant"))
    )
    if identity_hint:
        return "IDENTITY", None

    if is_numeric_dtype(series.dtype):
        return "NUMERIC", None

    return "TEXT", None


def markdown_escape(value: Any) -> str:
    return str(value).replace("|", r"\|").replace("\n", " ")


def display_value(value: Any) -> str:
    if pd.isna(value):
        return "N/A"
    if isinstance(value, pd.Timestamp):
        return value.isoformat(sep=" ")
    if isinstance(value, float):
        return f"{value:.10g}"
    return str(value)


def make_identity(source: Path, output_root: Path) -> IdentityResult:
    started_at = datetime.now().astimezone()
    charset_label, encoding = detect_charset(source)
    delimiter = detect_delimiter(source, encoding)
    frame = pd.read_csv(source, encoding=encoding, sep=delimiter)

    inferred: list[tuple[str, str]] = []
    statistics: list[tuple[str, str, Any, Any, Any, Any]] = []

    for column in frame.columns:
        column_name = str(column)
        series = frame[column]
        logical_type, parsed_temporal = infer_column_type(column_name, series)
        inferred.append((column_name, logical_type))

        if logical_type == "NUMERIC":
            numeric = pd.to_numeric(series, errors="coerce").dropna()
            if not numeric.empty:
                statistics.append(
                    (
                        column_name,
                        logical_type,
                        numeric.min(),
                        numeric.max(),
                        numeric.mean(),
                        numeric.median(),
                    )
                )
        elif logical_type in {"DATE", "DATETIME", "TIME"} and parsed_temporal is not None:
            temporal = parsed_temporal.dropna()
            if not temporal.empty:
                statistics.append(
                    (column_name, logical_type, temporal.min(), temporal.max(), None, None)
                )

    ended_at = datetime.now().astimezone()
    day_directory = output_root / started_at.strftime("%Y%m%d")
    day_directory.mkdir(parents=True, exist_ok=True)
    artifact = day_directory / f"{source.stem}_{started_at.strftime('%H%M%S')}.md"

    lines = [
        f"* Source: {source.resolve()}",
        f"* Filsize: {human_size(source.stat().st_size)}",
        f"* ProcessBeginAt: {started_at.strftime('%H:%M:%S')}",
        f"* ProcessEndAt: {ended_at.strftime('%H:%M:%S')}",
        f"* Charset: {charset_label}",
        f"* Rows Number: {len(frame)}",
        f"* Cols Number: {len(frame.columns)}",
        "",
        "## Column list",
        "",
        "| Colname | Infered Type |",
        "| --- | --- |",
    ]
    lines.extend(
        f"| {markdown_escape(column)} | {logical_type} |"
        for column, logical_type in inferred
    )

    lines.extend(
        [
            "",
            "## Numeric and temporal statistics",
            "",
            "| Col name | Infered Type | Min value | Max value | Mean value | Median value |",
            "| --- | --- | ---: | ---: | ---: | ---: |",
        ]
    )
    if statistics:
        for column, logical_type, minimum, maximum, mean, median in statistics:
            lines.append(
                "| "
                + " | ".join(
                    markdown_escape(display_value(value))
                    for value in (column, logical_type, minimum, maximum, mean, median)
                )
                + " |"
            )
    else:
        lines.append("| _Aucune colonne numérique ou temporelle_ | N/A | N/A | N/A | N/A | N/A |")

    artifact.write_text("\n".join(lines) + "\n", encoding="utf-8")
    return IdentityResult(source.resolve(), artifact.resolve(), len(frame), len(frame.columns))


def generate_identities(
    input_directory: Path = Path("datas"),
    output_root: Path = Path("artifacts/ingestion/identities"),
) -> list[IdentityResult]:
    sources = sorted(input_directory.glob("*.csv"))
    if not sources:
        raise FileNotFoundError(f"Aucun fichier CSV trouvé dans {input_directory.resolve()}")

    results: list[IdentityResult] = []
    for source in sources:
        try:
            results.append(make_identity(source, output_root))
        except Exception as exc:
            print(f"ERREUR pendant le traitement de {source.resolve()}")
            print(f"{type(exc).__name__}: {exc}")
            raise RuntimeError(f"Traitement interrompu sur {source.name}") from exc

    print(f"Traitement terminé avec succès : {len(results)} fichier(s) traité(s).")
    for result in results:
        print(
            f"- {result.source.name}: {result.rows} ligne(s), "
            f"{result.columns} colonne(s) -> {result.artifact}"
        )
    return results


if __name__ == "__main__":
    generate_identities()


# Carte d'identité du SQL
A l'aide de SQLAlchemy, il va s'agir de d'examiner les fichiers .sql.txt du dossier /datas, et pour chacun, générer leur carte d'identité dans un fichier artifacts/ingestion/identities/<AAAAMMJJ>/<nom du fichier analysé>_<HHMMSS>.md

Chaque fichier sera constitué de la manière suivante :
* Source: <chemin/complet/vers/le/document/source>
* Filsize : 9999.99 [Ko | Mo | ....]
* ProcessBeginAt: HH:MM:SS
* ProcessEndAt: HH:MM:SS
* Charset: [UTF8 | ISO8859 | Latin | ...]
* Rows Number : 9999
* Cols Number : 9999
* Column list (idéalement dans un tableau) | Colname | Infered Type | | --------| -------------| | machine_id | IDENTITY |

Pour chaque colonne identifiée numérique ou date :
* Col name,
* Min value,
* Max value,
* Mean value (uniquement pour les numériques),
* Median value (uniquement pour les numériques)

Affiche un compte-rendu en fin de traitement, et arrête-toi en cas d'erreur, en affichant le descriptif de l'erreur.

In [1]:
from pathlib import Path

from indusense.sql_identities import generate_sql_identities

results = generate_sql_identities(
    input_directory=Path("datas"),
    output_root=Path("artifacts/ingestion/identities"),
)

results

Traitement termine avec succes : 1 fichier(s) traite(s).
- machine.sql.txt: 2 table(s), 130 ligne(s), 22 colonne(s) -> D:\indusense\artifacts\ingestion\identities\20261120\machine_161317.md


[IdentityResult(source=WindowsPath('D:/indusense/datas/machine.sql.txt'), artifact=WindowsPath('D:/indusense/artifacts/ingestion/identities/20261120/machine_161317.md'), tables=2, rows=130, columns=22)]